# Budgerigar S1：已验证四层分级记忆与延迟复读
组合通过严格审计的语义短时记忆与 Human M0；完整时间轴没有 action/state 标签。

In [ ]:
#@title 1. 更新项目并连接 Drive
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib,shutil,datetime,json
repo=Path(REPO_DIR)
if not (repo/'.git').is_dir():subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else:
 pull=subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],text=True,capture_output=True);print(pull.stdout,pull.stderr)
 if pull.returncode:
  backup=repo.with_name(f'Budgerigar_backup_{datetime.datetime.now():%H%M%S}');shutil.move(str(repo),str(backup));subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train]'],check=True);sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]:del sys.modules[name]
importlib.invalidate_caches();from google.colab import drive;drive.mount('/content/drive');WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')

In [ ]:
#@title 2. 四层分级记忆结构与严格流式等价检查
import torch
if not torch.cuda.is_available():raise RuntimeError('请选择 GPU runtime')
from budgerigar.human_speech_model import HumanSpeechConfig
from budgerigar.streaming_short_memory import ShortMemoryConfig
from budgerigar.delayed_human_speech import DelayedHumanSpeechConfig,create_delayed_human_speech
semantic=ShortMemoryConfig(sample_rate=16000,tick_samples=160,hidden_dim=96,token_layers=4,attention_heads=4,acoustic_slots=0);config=DelayedHumanSpeechConfig(human=HumanSpeechConfig(),semantic=semantic);model=create_delayed_human_speech(config).cuda().eval();dummy=torch.randn(2,16,config.human.tick_samples,device='cuda')
with torch.no_grad():full=model(dummy)[0];state=None;parts=[]
with torch.no_grad():
 for index in range(dummy.shape[1]):value,state,_=model.stream_step(dummy[:,index],state);parts.append(value)
difference=float((full-torch.stack(parts,1)).abs().max());print('parameters:',sum(p.numel() for p in model.parameters()),'stream difference:',difference);assert difference<1e-4

In [ ]:
#@title 3. T4 课程训练：先适配记忆，再学习延迟波形
MAX_STEPS=700 #@param {type:'integer'}
BATCH_SIZE=6 #@param {type:'integer'}
MANIFEST=WORK_ROOT/'manifests'/'fsdd.jsonl';HUMAN_CHECKPOINT=WORK_ROOT/'checkpoints'/'human_cochlear_perceptual_m0'/'best.pt';SEMANTIC_CHECKPOINT=WORK_ROOT/'checkpoints'/'streaming_hierarchical_memory_decoder_s0'/'best.pt';EVALUATOR_CHECKPOINT=WORK_ROOT/'checkpoints'/'frozen_real_audio_digit_evaluator'/'best.pt';RUN_DIR=WORK_ROOT/'checkpoints'/'delayed_verified_hierarchical_memory_human_s1_curriculum'
assert HUMAN_CHECKPOINT.is_file() and SEMANTIC_CHECKPOINT.is_file() and EVALUATOR_CHECKPOINT.is_file()
from budgerigar.train_delayed_human import DelayedHumanTrainingConfig,train_delayed_human
training=DelayedHumanTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS,semantic_warmup_steps=250,max_train_records=1000,max_validation_records=100,thinking_ms=(140,220),semantic_learning_rate_scale=.1)
report=train_delayed_human(MANIFEST,RUN_DIR,HUMAN_CHECKPOINT,SEMANTIC_CHECKPOINT,EVALUATOR_CHECKPOINT,training,config);print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 4. 完整时间轴试听与 smoke 门槛
best=min(report['history'],key=lambda x:x['validation_repeat_acoustic_loss']+1-x['validation_output_digit_accuracy']+5*x['validation_early_output_rms'])
s1_pass=best['validation_output_digit_accuracy']>.7 and best['validation_semantic_end_accuracy']>.7 and best['validation_semantic_delay_accuracy']>.7 and best['validation_semantic_cleared_accuracy']<.2 and best['validation_early_output_rms']<.01 and best['validation_onset_mae_ms']<100
print(json.dumps(best,ensure_ascii=False,indent=2));print('s1_pass =',s1_pass)
payload=torch.load(RUN_DIR/'best_validation_example.pt',map_location='cpu',weights_only=False);import torchaudio
for key in ('input','target','output'):torchaudio.save(str(RUN_DIR/f's1_{key}.wav'),payload[key].unsqueeze(0),payload['sample_rate'])
from IPython.display import Audio,display
print('输入时间轴（数字后为静默）：');display(Audio(filename=str(RUN_DIR/'s1_input.wav')))
print('目标时间轴（思考后复读）：');display(Audio(filename=str(RUN_DIR/'s1_target.wav')))
print('模型完整输出：');display(Audio(filename=str(RUN_DIR/'s1_output.wav')))
if not s1_pass:print('未通过：不要解冻发声器或扩大到长句，先检查语义适配、记忆清空、过早发声和起始误差。')